# Week 4 – Feature Engineering & Baseline Modelling

**Project:** Forecasting Model Development: AI-Powered Energy Prediction  
**Environment:** VS Code + Jupyter Notebook  

This notebook is prepared for the Week 4 tasks: data inspection, preprocessing, time-based features, lag features, baseline modelling, and evaluation.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

print('Libraries imported successfully.')

## 2. Project configuration

Update these three values after you choose your dataset.

In [ ]:
# Change these values to match your dataset
DATA_FILE = '../data/raw/your_dataset.csv'
TIMESTAMP_COLUMN = 'timestamp'
TARGET_COLUMN = 'load'

print('Dataset:', DATA_FILE)
print('Timestamp column:', TIMESTAMP_COLUMN)
print('Target column:', TARGET_COLUMN)

## 3. Load the dataset

In [ ]:
df = pd.read_csv(DATA_FILE)
print('Dataset shape:', df.shape)
df.head()

## 4. Basic data inspection

In [ ]:
print('Columns:')
print(df.columns.tolist())

print('\nMissing values:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

df.describe(include='all')

## 5. Prepare the timestamp

For forecasting, the data must be ordered by time.

In [ ]:
df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN], errors='coerce')
df = df.dropna(subset=[TIMESTAMP_COLUMN])
df = df.sort_values(TIMESTAMP_COLUMN).reset_index(drop=True)

df[[TIMESTAMP_COLUMN, TARGET_COLUMN]].head()

## 6. Handle missing target values

This example uses time-based interpolation. The method can be changed after inspecting the selected dataset.

In [ ]:
print('Missing target values before:', df[TARGET_COLUMN].isnull().sum())

df[TARGET_COLUMN] = df[TARGET_COLUMN].interpolate(method='linear')

print('Missing target values after:', df[TARGET_COLUMN].isnull().sum())

## 7. Create time-based features

These features help the model learn daily, weekly, and seasonal patterns.

In [ ]:
df['hour'] = df[TIMESTAMP_COLUMN].dt.hour
df['day'] = df[TIMESTAMP_COLUMN].dt.day
df['day_of_week'] = df[TIMESTAMP_COLUMN].dt.dayofweek
df['month'] = df[TIMESTAMP_COLUMN].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df[[TIMESTAMP_COLUMN, 'hour', 'day', 'day_of_week', 'month', 'is_weekend']].head()

## 8. Create lag features

`lag_1` uses the previous observation. `lag_24` is suitable for hourly data because it represents the same time on the previous day. If your dataset uses a different interval, we will adjust this.

In [ ]:
df['lag_1'] = df[TARGET_COLUMN].shift(1)
df['lag_24'] = df[TARGET_COLUMN].shift(24)

# Remove rows made incomplete by lagging
df_model = df.dropna().copy()

df_model[[TARGET_COLUMN, 'lag_1', 'lag_24']].head()

## 9. Visualise the target over time

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df[TIMESTAMP_COLUMN], df[TARGET_COLUMN])
plt.xlabel('Time')
plt.ylabel(TARGET_COLUMN)
plt.title('Energy Load Over Time')
plt.tight_layout()
plt.show()

## 10. Check feature correlation

In [ ]:
feature_columns = ['hour', 'day', 'day_of_week', 'month', 'is_weekend', 'lag_1', 'lag_24']

correlations = df_model[feature_columns + [TARGET_COLUMN]].corr()[TARGET_COLUMN].sort_values(ascending=False)
correlations

## 11. Chronological train/test split

Time-series data should not be randomly shuffled. The earlier observations are used for training and the later observations for testing.

In [ ]:
X = df_model[feature_columns]
y = df_model[TARGET_COLUMN]

split_index = int(len(df_model) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 12. Train a Linear Regression baseline

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

print('Linear Regression trained successfully.')

## 13. Evaluate the baseline model

In [ ]:
mae = mean_absolute_error(y_test, linear_predictions)
rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))

# MAPE is calculated only where actual values are not zero.
non_zero = y_test != 0
mape = np.mean(np.abs((y_test[non_zero] - linear_predictions[non_zero]) / y_test[non_zero])) * 100

print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAPE: {mape:.2f}%')

## 14. Actual vs predicted values

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(y_test.values[:200], label='Actual')
plt.plot(linear_predictions[:200], label='Predicted')
plt.xlabel('Test observation')
plt.ylabel(TARGET_COLUMN)
plt.title('Linear Regression: Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

## 15. Residual analysis

In [ ]:
residuals = y_test.values - linear_predictions

plt.figure(figsize=(10, 4))
plt.scatter(linear_predictions, residuals, s=10)
plt.axhline(0)
plt.xlabel('Predicted Load')
plt.ylabel('Residual')
plt.title('Linear Regression Residual Plot')
plt.tight_layout()
plt.show()

## 16. Week 4 notes / findings

Complete this section after running the notebook:

- Dataset selected: **[enter dataset name]**
- Target variable: **[enter target]**
- Missing values found: **[enter result]**
- Time-based features created: **hour, day, day_of_week, month, is_weekend**
- Lag features created: **lag_1, lag_24**
- Linear Regression MAE: **[enter result]**
- Linear Regression RMSE: **[enter result]**
- Linear Regression MAPE: **[enter result]**
- Main observation: **[enter finding]**

### Next step
Train XGBoost and compare its performance with the Linear Regression baseline.